# 01 — Download GSS cumulative data

Downloads the GSS 1972–2024 cumulative cross-sectional file (Stata format,
Release 3a) from NORC into `../data/raw/gss/`, extracts the handful of
variables needed for the Jewish identity trend analysis, and saves them to
`../data/derived/gss_jewish_identity.parquet`.

Variables kept:

| variable | label |
|----------|-------|
| `year`, `id` | survey year, respondent ID |
| `age` | age in years (topcoded at 89) |
| `relig` | R's religious preference |
| `relig16` | religion in which raised (at age 16) |
| `jew`, `jew16` | Jewish denomination, current / at 16 |
| `wtssps` | person post-stratification weight (defined for **all** years, including 2021+) |


In [1]:
import os
import zipfile
import urllib.request

DATA = os.path.abspath(os.path.join('..', 'data'))
RAW = os.path.join(DATA, 'raw', 'gss')
DERIVED = os.path.join(DATA, 'derived')
os.makedirs(RAW, exist_ok=True)
os.makedirs(DERIVED, exist_ok=True)

URL = 'https://gss.norc.org/content/dam/gss/get-the-data/documents/stata/GSS_stata.zip'
zip_path = os.path.join(RAW, 'GSS_stata.zip')
dta_path = os.path.join(RAW, 'GSS_stata', 'gss7224_r3a.dta')

if not os.path.exists(dta_path):
    if not os.path.exists(zip_path):
        print('downloading', URL)
        urllib.request.urlretrieve(URL, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(RAW)
print(f'{os.path.getsize(dta_path)/1e6:.0f} MB', dta_path)

598 MB /home/user/ai_assisted_us_health_data_analysis/data/raw/gss/GSS_stata/gss7224_r3a.dta


In [2]:
import warnings
import pandas as pd

warnings.filterwarnings('ignore', message='.*could not be decoded.*')

cols = ['year', 'id', 'age', 'relig', 'relig16', 'jew', 'jew16', 'wtssps']
df = pd.read_stata(dta_path, columns=cols, convert_categoricals=True)
df['year'] = df['year'].astype(int)
# age is categorical with an '89 or older' topcode; make it numeric (89 = 89+)
df['age'] = pd.to_numeric(df['age'].astype(str).replace('89 or older', '89'),
                          errors='coerce')
print(df.shape)
df.head()

/tmp/ipykernel_3284/1903951603.py:7: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  df = pd.read_stata(dta_path, columns=cols, convert_categoricals=True)


(75699, 8)


,year,id,age,relig,relig16,jew,jew16,wtssps
0,1972,1,23.0,jewish,NaN,NaN,NaN,0.663196
1,1972,2,70.0,catholic,NaN,NaN,NaN,0.917370
2,1972,3,48.0,protestant,NaN,NaN,NaN,0.897413
3,1972,4,27.0,other,NaN,NaN,NaN,1.066341
4,1972,5,61.0,protestant,NaN,NaN,NaN,0.944324


In [3]:
out_path = os.path.join(DERIVED, 'gss_jewish_identity.parquet')
df.to_parquet(out_path)

# quick sanity check: coverage and raw Jewish counts by decade
chk = df.assign(decade=df.year // 10 * 10).groupby('decade').agg(
    n=('id', 'size'),
    n_jewish=('relig', lambda s: (s == 'jewish').sum()),
    n_raised_jewish=('relig16', lambda s: (s == 'jewish').sum()),
    wt_missing=('wtssps', lambda s: s.isna().sum()),
)
chk

,n,n_jewish,n_raised_jewish,wt_missing
decade,,,,
1970,10652,254,218,0
1980,14241,284,302,0
1990,13223,269,272,0
2000,14927,283,275,0
2010,11771,195,199,0
2020,10885,183,189,0
